# 02 — Text Cleaning & Section Extraction
Parse downloaded 10-K HTML filings and extract Item 1A (Risk Factors) 
and Item 7 (MD&A) sections for downstream NLP analysis.

In [4]:
import os
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

RAW_DIR = Path("../data/raw/sec-edgar-filings")
CLEANED_DIR = Path("../data/cleaned")
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

print("Ready.")

Ready.


In [5]:
def extract_sections(filepath):
    """Extract Item 1A and Item 7 text from a full-submission.txt 10-K filing."""
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    
    # Strip HTML tags if present
    soup = BeautifulSoup(text, "lxml")
    text = soup.get_text(separator=" ")
    text = re.sub(r'\s+', ' ', text).strip()

    sections = {"item_1a": "", "item_7": ""}

    # Item 1A - Risk Factors
    match_1a = re.search(
        r'item\s+1a[\.\s]*risk factors(.+?)item\s+1b',
        text, re.IGNORECASE | re.DOTALL
    )
    if match_1a:
        sections["item_1a"] = match_1a.group(1).strip()[:50000]

    # Item 7 - MD&A
    match_7 = re.search(
        r'item\s+7[\.\s]*management.{0,50}discussion(.+?)item\s+7a',
        text, re.IGNORECASE | re.DOTALL
    )
    if match_7:
        sections["item_7"] = match_7.group(1).strip()[:50000]

    return sections

In [6]:
records = []

companies = [d for d in RAW_DIR.iterdir() if d.is_dir()]
print(f"Processing {len(companies)} companies...\n")

for company_dir in sorted(companies):
    ticker = company_dir.name
    filing_dir = company_dir / "10-K"
    
    if not filing_dir.exists():
        continue

    for filing in sorted(filing_dir.iterdir()):
        filepath = filing / "full-submission.txt"
        
        if not filepath.exists():
            continue
        
        try:
            sections = extract_sections(filepath)
            
            # Extract year from folder name format: 0000320193-17-000070
            year_match = re.search(r'^\d{10}-(\d{2})-', filing.name)
            if year_match:
                two_digit = int(year_match.group(1))
                year = 2000 + two_digit
            else:
                year = None
            
            records.append({
                "ticker": ticker,
                "year": year,
                "item_1a": sections["item_1a"],
                "item_7": sections["item_7"],
                "item_1a_words": len(sections["item_1a"].split()),
                "item_7_words": len(sections["item_7"].split()),
            })
            print(f"✓ {ticker} {year}")
        except Exception as e:
            print(f"✗ {ticker} {filing.name}: {e}")

df = pd.DataFrame(records)
print(f"\nExtracted {len(df)} filings")
print(f"Item 1A extracted: {(df['item_1a_words'] > 0).sum()}")
print(f"Item 7 extracted:  {(df['item_7_words'] > 0).sum()}")

Processing 51 companies...

✓ AAPL 2017
✓ AAPL 2018
✓ AAPL 2019
✓ AAPL 2020
✓ AAPL 2021
✓ AAPL 2022
✓ AAPL 2023
✓ AAPL 2015
✓ AAPL 2016
✓ ABBV 2015
✓ ABBV 2016
✓ ABBV 2017
✓ ABBV 2018
✓ ABBV 2019
✓ ABBV 2020
✓ ABBV 2021
✓ ABBV 2022
✓ ABBV 2023
✓ ABT 2015
✓ ABT 2016
✓ ABT 2017
✓ ABT 2018
✓ ABT 2019
✓ ABT 2020
✓ ABT 2021
✓ ABT 2022
✓ ABT 2023
✓ ADBE 2015
✓ ADBE 2016
✓ ADBE 2017
✓ ADBE 2018
✓ ADBE 2019
✓ ADBE 2020
✓ ADBE 2021
✓ ADBE 2022
✓ ADBE 2023
✓ AMD 2016
✓ AMD 2017
✓ AMD 2018
✓ AMD 2019
✓ AMD 2020
✓ AMD 2022
✓ AMD 2023
✓ AMD 2015
✓ AMD 2021
✓ AMZN 2015
✓ AMZN 2016
✓ AMZN 2017
✓ AMZN 2018
✓ AMZN 2019
✓ AMZN 2020
✓ AMZN 2021
✓ AMZN 2022
✓ AMZN 2023
✓ APD 2017
✓ APD 2018
✓ APD 2019
✓ APD 2020
✓ APD 2021
✓ APD 2022
✓ APD 2023
✓ APD 2015
✓ APD 2016
✓ AVGO 2018
✓ AVGO 2019
✓ AVGO 2020
✓ AVGO 2021
✓ AVGO 2022
✓ AVGO 2023
✓ BAC 2015
✓ BAC 2016
✓ BAC 2017
✓ BAC 2018
✓ BAC 2019
✓ BAC 2020
✓ BAC 2021
✓ BAC 2022
✓ BAC 2023
✓ BRK-B 2023
✓ BRK-B 2015
✓ BRK-B 2016
✓ BRK-B 2017
✓ BRK-B 2018
✓ BRK-B

In [7]:
from pathlib import Path
CLEANED_DIR = Path("../data/cleaned")

output_path = CLEANED_DIR / "extracted_sections.csv"
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")
print()

print(df[["ticker", "year", "item_1a_words", "item_7_words"]].sort_values(
    ["ticker", "year"]).head(20).to_string(index=False))

Saved to ../data/cleaned/extracted_sections.csv

ticker  year  item_1a_words  item_7_words
  AAPL  2015              1            10
  AAPL  2016              1            10
  AAPL  2017              1            10
  AAPL  2018              1            10
  AAPL  2019              1            10
  AAPL  2020              1            10
  AAPL  2021              1            10
  AAPL  2022              1            10
  AAPL  2023              1            10
  ABBV  2015           7236          8124
  ABBV  2016           6964          8190
  ABBV  2017              1            10
  ABBV  2018              1            10
  ABBV  2019              1            10
  ABBV  2020              1            10
  ABBV  2021              1            10
  ABBV  2022              1            10
  ABBV  2023              1            10
   ABT  2015           3659          7536
   ABT  2016           3813          7632
